In [ ]:
!pip -q install yacs scipy scikit-image imageio imageio-ffmpeg dominate dill opencv-python-headless matplotlib ninja lmdb
!git clone https://github.com/RenYurui/PIRender.git /content/PIRender
%cd /content/PIRender
!git submodule update --init --recursive
!pip -q install -r requirements.txt
print("repo ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 34.6 MB/s eta 0:00:00
Cloning into '/content/PIRender'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 147 (delta 41), reused 126 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 197.67 KiB | 16.47 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/PIRender
Submodule 'Deep3DFaceRecon_pytorch' (https://github.com/sicxu/Deep3DFaceRecon_pytorch) registered for path 'Deep3DFaceRecon_pytorch'
Cloning into '/content/PIRender/Deep3DFaceRecon_pytorch'...
Submodule path 'Deep3DFaceRecon_pytorch': checked out '73d491102af6731bded9ae6b3cc7466c3b2e9e48'
Submodule 'nvdiffrast' (https://github.com/NVlabs/nvdiffrast.git) registered for path 'Deep3DFaceRecon_pytorch/nvdiffrast'
Cloning into '/content/PIRender/Deep3DFaceRecon_pyt

In [ ]:
%cd /content/PIRender
!bash scripts/download_weights.sh
!cp scripts/face_recon_videos.py ./Deep3DFaceRecon_pytorch
!cp scripts/extract_kp_videos.py ./Deep3DFaceRecon_pytorch
!cp scripts/coeff_detector.py ./Deep3DFaceRecon_pytorch
!cp scripts/inference_options.py ./Deep3DFaceRecon_pytorch/options
print("weights prepared")


/content/PIRender
Downloading...
From (original): https://drive.google.com/uc?id=1-0xOf6g58OmtKtEWJlU3VlnfRqPN9Uq7
From (redirected): https://drive.google.com/uc?id=1-0xOf6g58OmtKtEWJlU3VlnfRqPN9Uq7&confirm=t&uuid=43af38a5-86c1-4117-954f-f7163e80eceb
To: /content/PIRender/face.zip
100% 167M/167M [00:06<00:00, 24.7MB/s]
Archive:  ./face.zip
   creating: face/
  inflating: face/epoch_00190_iteration_000400000_checkpoint.pt  
  inflating: face/latest_checkpoint.txt  
weights prepared


In [ ]:
import os
import cv2
import glob
import shutil
import imageio
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

ROOT = "/content/pirender_demo"
SOURCE_DIR = f"{ROOT}/source"
DRIVING_VIDEO_DIR = f"{ROOT}/driving_video"
DRIVING_FRAMES_DIR = f"{ROOT}/driving_frames"
SOURCE_REPEAT_DIR = f"{ROOT}/source_repeat"
PAIR_DIR = f"{ROOT}/pairs"
PAIR_COEFF_DIR = f"{PAIR_DIR}/coeff"
RESULTS_DIR = f"{ROOT}/results"

for path in [SOURCE_DIR, DRIVING_VIDEO_DIR, DRIVING_FRAMES_DIR, SOURCE_REPEAT_DIR, PAIR_DIR, PAIR_COEFF_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)


In [ ]:
print("Upload source image")
source_upload = files.upload()
source_name = next(iter(source_upload.keys()))
source_path = f"{SOURCE_DIR}/source.png"
shutil.move(source_name, source_path)

print("Upload driving video")
driving_upload = files.upload()
driving_name = next(iter(driving_upload.keys()))
driving_video_path = f"{DRIVING_VIDEO_DIR}/driving.mp4"
shutil.move(driving_name, driving_video_path)

cap = cv2.VideoCapture(driving_video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
count = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    cv2.imwrite(f"{DRIVING_FRAMES_DIR}/frame_{count:05d}.png", frame)
    shutil.copy(source_path, f"{SOURCE_REPEAT_DIR}/frame_{count:05d}.png")
    count += 1
cap.release()

print("frames extracted:", count, "fps:", fps)


Upload source image


Saving ek.jpeg to ek.jpeg
Upload driving video


Saving result_faceswap.mp4 to result_faceswap.mp4
frames extracted: 78 fps: 25.0


In [ ]:
%cd /content/PIRender/Deep3DFaceRecon_pytorch

!rm -rf /content/insightface
!git clone https://github.com/deepinsight/insightface.git /content/insightface
!cp -r /content/insightface/recognition/arcface_torch ./models/

import os
assert os.path.isdir("/content/PIRender/Deep3DFaceRecon_pytorch/models/arcface_torch"), "arcface_torch copy failed"

print("arcface_torch ready")

%cd /content/PIRender/Deep3DFaceRecon_pytorch
!pip -q install kornia
!pip -q install kornia yacs scipy scikit-image imageio imageio-ffmpeg dominate dill ninja lmdb tensorboard



/content/PIRender/Deep3DFaceRecon_pytorch
Cloning into '/content/insightface'...
remote: Enumerating objects: 12667, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 12667 (delta 18), reused 19 (delta 0), pack-reused 12628 (from 1)
Receiving objects: 100% (12667/12667), 58.43 MiB | 12.15 MiB/s, done.
Resolving deltas: 100% (6615/6615), done.
arcface_torch ready
/content/PIRender/Deep3DFaceRecon_pytorch


In [ ]:
%cd /content/PIRender/Deep3DFaceRecon_pytorch

import pathlib

p = pathlib.Path("/content/PIRender/Deep3DFaceRecon_pytorch/util/preprocess.py")
text = p.read_text()

text = text.replace(
    'warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)',
    'warnings.filterwarnings("ignore", category=DeprecationWarning)'
)

p.write_text(text)
print("patched preprocess.py")


/content/PIRender/Deep3DFaceRecon_pytorch
patched preprocess.py


In [ ]:
%cd /content/PIRender/Deep3DFaceRecon_pytorch
!pip -q install face-alignment


/content/PIRender/Deep3DFaceRecon_pytorch


In [ ]:
!pip -q install face-alignment kornia yacs scipy scikit-image imageio imageio-ffmpeg dominate dill ninja lmdb tensorboard

In [ ]:
%cd /content/PIRender/Deep3DFaceRecon_pytorch

!pip -q install setuptools wheel ninja
!pip -q install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation


/content/PIRender/Deep3DFaceRecon_pytorch
  Preparing metadata (pyproject.toml) ... done


In [ ]:
%cd /content/PIRender/Deep3DFaceRecon_pytorch

!python coeff_detector.py \
    --input_dir {DRIVING_FRAMES_DIR} \
    --keypoint_dir {DRIVING_FRAMES_DIR} \
    --output_dir {PAIR_COEFF_DIR} \
    --name=model_name \
    --epoch=20 \
    --model facerecon

import glob
coeffs = sorted(glob.glob(f"{PAIR_COEFF_DIR}/*_3dmm_coeff.txt"))
print("coeff files:", len(coeffs))
assert len(coeffs) > 0, "coeff extraction failed"
print("driving coeff extraction done")


/content/PIRender/Deep3DFaceRecon_pytorch
Traceback (most recent call last):
  File "/content/PIRender/Deep3DFaceRecon_pytorch/coeff_detector.py", line 94, in <module>
    opt = InferenceOptions().parse() 
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/PIRender/Deep3DFaceRecon_pytorch/options/base_options.py", line 124, in parse
    opt = self.gather_options()
          ^^^^^^^^^^^^^^^^^^^^^
  File "/content/PIRender/Deep3DFaceRecon_pytorch/options/base_options.py", line 73, in gather_options
    model_option_setter = models.get_option_setter(model_name)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/PIRender/Deep3DFaceRecon_pytorch/models/__init__.py", line 50, in get_option_setter
    model_class = find_model_using_name(model_name)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/PIRender/Deep3DFaceRecon_pytorch/models/__init__.py", line 33, in find_model_using_name
    modellib = importlib.import_module(model_filename)
    

AssertionError: coeff extraction failed

In [ ]:
import glob
import shutil
driving_coeffs = sorted(glob.glob(f"{PAIR_COEFF_DIR}/*_3dmm_coeff.txt"))
assert driving_coeffs, "No driving coeffs found."

for idx, coeff_path in enumerate(driving_coeffs):
    stem = f"frame_{idx:05d}"
    shutil.copy(source_path, f"{PAIR_DIR}/{stem}.png")
    shutil.copy(coeff_path, f"{PAIR_DIR}/{stem}_3dmm_coeff.txt")

print("paired samples:", len(driving_coeffs))


AssertionError: No driving coeffs found.

In [ ]:
%cd /content/PIRender
import os
import torch
from types import SimpleNamespace
from config import Config
from data.image_dataset import ImageDataset
from util.logging import init_logging, make_logging_dir
from util.trainer import get_model_optimizer_and_scheduler, get_trainer, set_random_seed

set_random_seed(0)

args = SimpleNamespace(
    config="./config/face_demo.yaml",
    name="face",
    checkpoints_dir="result",
    seed=0,
    which_iter=None,
    no_resume=True,
    output_dir=RESULTS_DIR,
    local_rank=0,
    single_gpu=True,
)

opt = Config(args.config, args, is_train=False)
opt.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
date_uid, logdir = init_logging(opt)
opt.logdir = logdir
make_logging_dir(logdir, date_uid)

net_G, net_G_ema, opt_G, sch_G = get_model_optimizer_and_scheduler(opt)
trainer = get_trainer(opt, net_G, net_G_ema, opt_G, sch_G, None)
current_epoch, current_iteration = trainer.load_checkpoint(opt, args.which_iter)
net_G = trainer.net_G_ema.eval()

dataset = ImageDataset(opt.data, PAIR_DIR)
frames = []
with torch.no_grad():
    for _ in range(len(dataset)):
        data = dataset.next_image()
        source = data["source_image"].to(opt.device)
        target = data["target_semantics"].to(opt.device)
        output = net_G(source, target)
        fake = output["fake_image"].cpu().clamp_(-1, 1)[0]
        fake = ((fake.permute(1, 2, 0).numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        frames.append(fake)

print("generated frames:", len(frames), "checkpoint epoch:", current_epoch, "iter:", current_iteration)


In [ ]:
OUTPUT_VIDEO = f"{ROOT}/pirender_output.mp4"
writer = imageio.get_writer(OUTPUT_VIDEO, fps=fps)
for frame in frames:
    writer.append_data(frame)
writer.close()

print("video saved:", OUTPUT_VIDEO)
files.download(OUTPUT_VIDEO)


In [ ]:
sample_ids = np.linspace(0, len(frames) - 1, num=min(5, len(frames)), dtype=int)
plt.figure(figsize=(15, 3))
for i, idx in enumerate(sample_ids, 1):
    plt.subplot(1, len(sample_ids), i)
    plt.imshow(frames[idx])
    plt.axis("off")
    plt.title(idx)
plt.tight_layout()
plt.show()
